# Assignment 11B – Reinforcement Learning: Taxi Problem

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

ModuleNotFoundError: No module named 'gymnasium'

## 1. Set Up Environment

In [ ]:
env = gym.make('Taxi-v3')

N_STATES  = env.observation_space.n    # 500
N_ACTIONS = env.action_space.n         # 6
print(f'States: {N_STATES} | Actions: {N_ACTIONS}')

## 2. Q-Learning

In [ ]:
Q = np.zeros((N_STATES, N_ACTIONS))

ALPHA     = 0.1
GAMMA     = 0.95
EPSILON   = 1.0
EPS_DECAY = 0.995
EPS_MIN   = 0.01
EPISODES  = 2000

rewards_per_episode = []

for ep in range(EPISODES):
    state, _ = env.reset()
    total_r  = 0

    for _ in range(200):
        if np.random.rand() < EPSILON:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[state])

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        Q[state, action] += ALPHA * (reward + GAMMA * np.max(Q[next_state]) - Q[state, action])

        state    = next_state
        total_r += reward
        if done:
            break

    EPSILON = max(EPS_MIN, EPSILON * EPS_DECAY)
    rewards_per_episode.append(total_r)

print('Training done.')

## 3. Training Rewards

In [ ]:
smoothed = np.convolve(rewards_per_episode, np.ones(50)/50, mode='valid')

plt.figure(figsize=(9, 3))
plt.plot(rewards_per_episode, alpha=0.3, color='steelblue', label='Raw')
plt.plot(range(49, EPISODES), smoothed, color='tomato', label='50-ep avg')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Q-Learning – Reward per Episode (Taxi)')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Test – Greedy Policy

In [ ]:
state, _ = env.reset()
total_r  = 0
steps    = 0

for _ in range(50):
    action = np.argmax(Q[state])
    state, reward, terminated, truncated, _ = env.step(action)
    total_r += reward
    steps   += 1
    if terminated or truncated:
        break

print(f'Test episode — Steps: {steps} | Total Reward: {total_r}')
env.close()

## 5. Evaluate over 100 Test Episodes

In [ ]:
env = gym.make('Taxi-v3')
test_rewards, test_steps = [], []

for _ in range(100):
    state, _ = env.reset()
    total_r = steps = 0
    for _ in range(200):
        action = np.argmax(Q[state])
        state, reward, terminated, truncated, _ = env.step(action)
        total_r += reward
        steps   += 1
        if terminated or truncated:
            break
    test_rewards.append(total_r)
    test_steps.append(steps)

env.close()
print(f'Avg Reward: {np.mean(test_rewards):.2f} | Avg Steps: {np.mean(test_steps):.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(test_rewards, bins=20, color='steelblue', edgecolor='k', linewidth=0.4)
axes[0].set_title('Test Reward Distribution')
axes[0].set_xlabel('Total Reward')

axes[1].hist(test_steps, bins=20, color='seagreen', edgecolor='k', linewidth=0.4)
axes[1].set_title('Steps to Complete')
axes[1].set_xlabel('Steps')

plt.suptitle('Taxi Q-Learning – 100 Test Episodes')
plt.tight_layout()
plt.show()